In [ ]:
#!/usr/bin/env python3

"""
run_1nn_accuracy_table.py

Driver script for reproducing the 1-NN classification accuracy table
for the six DNA representation methods.

The script:
1. Loads each NCBI dataset metadata file.
2. Loads the precomputed distance matrix for each method.
3. Performs leave-one-out 1-NN classification.
4. Computes accuracy.
5. Displays a clean pandas table in Jupyter.
6. Saves the numerical results to CSV.

Methods:
    CAKR, NVM, FFP-JS, FFP-KL, FPS, MKS

"""

import os
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder


# ============================================================
# Leave-one-out 1-NN from precomputed distance matrix
# ============================================================

def leave_one_out_1nn(distance_matrix, labels):
    """
    Perform leave-one-out 1-NN classification.
    """

    N = distance_matrix.shape[0]
    y_pred = np.zeros(N, dtype=labels.dtype)

    for i in range(N):
        dist = distance_matrix[i].copy()
        dist[i] = np.inf
        nearest_index = np.argmin(dist)
        y_pred[i] = labels[nearest_index]

    return y_pred


# ============================================================
# Label filtering
# ============================================================

def find_labels_with_min_count(labels, min_count=3):
    """
    Return indices of samples whose labels occur at least min_count times.
    """

    label_counts = Counter(labels)

    valid_labels = {
        label for label, count in label_counts.items()
        if count >= min_count
    }

    indices = [
        i for i, label in enumerate(labels)
        if label in valid_labels
    ]

    return indices


# ============================================================
# Distance-file paths
# ============================================================

def get_distance_path(method_key, data_name, k):
    """
    Return the distance-matrix path for each method.

    Adjust this function only if your repository uses different names.
    """

    if method_key == "CAKR":
        encoder_folder = "CAKR"
        return f"distances/{encoder_folder}/{data_name}/k{k}_distance_facet0.npy"

    if method_key == "NVM":
        return f"distances/NVM/{data_name}/k{k}_distance.npy"

    if method_key == "FFP-JS":
        return f"distances/FFP-JS/{data_name}/k{k}_distance.npy"

    if method_key == "FFP-KL":
        return f"distances/FFP-KL/{data_name}/k{k}_distance.npy"

    if method_key == "MKS":
        return f"distances/MKS/{data_name}/k{k}_distance.npy"

    if method_key == "FPS":
        path1 = f"distances/FPS/{data_name}/distance.npy"
        path2 = f"distances/FPS/{data_name}/k{k}_distance.npy"

        if os.path.exists(path1):
            return path1
        return path2

    raise ValueError(f"Unknown method: {method_key}")


# ============================================================
# Configuration
# ============================================================

datasets = {
    "NCBI 2020": "Yau2020_record_processed",
    "NCBI 2022": "Yau2022_record_processed",
    "NCBI 2024": "NCBI_record_valid_nucleotide",
    "NCBI 2024 All": "NCBI_record_valid_count",
}

methods = [
    "CAKR",
    "NVM",
    "FFP-JS",
    "FFP-KL",
    "FPS",
    "MKS",
]

method_k = {
    "CAKR": 4,
    "NVM": 5,
    "FFP-JS": 3,
    "FFP-KL": 3,
    "FPS": 3,
    "MKS": 3,
}

min_count = 3

output_csv = "one_nn_accuracy_table_results.csv"


# ============================================================
# Load labels
# ============================================================

def load_dataset_labels(data_name, min_count=3):
    """
    Load accessions and labels, sort by accession, filter by min_count,
    and return filtered indices and integer labels.
    """

    csv_path = f"data/{data_name}.csv"

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing dataset CSV: {csv_path}")

    df = pd.read_csv(csv_path)

    x_list = df["Accession (version)"].to_list()
    y_list = df["Family"].to_list()

    # Sort to match the order used when distance matrices were created
    x_list, y_list = zip(*sorted(zip(x_list, y_list)))
    x_list = list(x_list)
    y_list = list(y_list)

    indices = find_labels_with_min_count(y_list, min_count=min_count)

    y_filtered = [y_list[i] for i in indices]

    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(y_filtered)
    labels = np.array(labels, dtype=np.int32)

    return indices, labels, x_list, y_list


# ============================================================
# Main computation
# ============================================================

results = []

for dataset_label, data_name in datasets.items():
    print("\n===================================================")
    print(f"Dataset: {dataset_label}")
    print(f"Data file key: {data_name}")
    print("===================================================")

    try:
        indices, labels, x_list, y_list = load_dataset_labels(
            data_name=data_name,
            min_count=min_count,
        )
    except FileNotFoundError as e:
        print(e)
        continue

    print("Original sample size:", len(y_list))
    print("Filtered sample size:", len(labels))
    print("Number of classes:", len(np.unique(labels)))

    for method in methods:
        k = method_k[method]
        distance_path = get_distance_path(method, data_name, k)

        print(f"\nMethod: {method}")
        print(f"k: {k}")
        print("Distance path:", distance_path)

        if not os.path.exists(distance_path):
            print("Missing distance file. Skipping.")
            results.append({
                "Data": dataset_label,
                "Method": method,
                "k": k,
                "Accuracy": np.nan,
                "Distance_file": distance_path,
                "Status": "missing_distance_file",
            })
            continue

        distance_matrix = np.load(distance_path)

        if distance_matrix.shape[0] != len(y_list):
            print("Shape mismatch. Skipping.")
            print("Distance matrix shape:", distance_matrix.shape)
            print("Number of labels before filtering:", len(y_list))

            results.append({
                "Data": dataset_label,
                "Method": method,
                "k": k,
                "Accuracy": np.nan,
                "Distance_file": distance_path,
                "Status": "shape_mismatch",
            })
            continue

        subD = distance_matrix[np.ix_(indices, indices)]

        y_pred = leave_one_out_1nn(subD, labels)

        accuracy = accuracy_score(labels, y_pred)

        print(f"Accuracy: {accuracy:.4f}")

        results.append({
            "Data": dataset_label,
            "Method": method,
            "k": k,
            "Accuracy": accuracy,
            "Distance_file": distance_path,
            "Status": "ok",
        })


# ============================================================
# Save raw results
# ============================================================

results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print("\nSaved numerical results to:")
print(output_csv)


# ============================================================
# Make wide Jupyter table
# ============================================================

wide = results_df.pivot(
    index="Data",
    columns="Method",
    values="Accuracy",
)

# Keep manuscript row and column order
wide = wide.loc[list(datasets.keys()), methods]

# Round as in manuscript
wide_rounded = wide.round(3)

print("\n===================================================")
print("1-NN accuracy table")
print("===================================================")

display(wide_rounded)